# PSG backbone：逐步调试

从上到下用 **Shift+Enter** 执行。每个变量保留在 kernel 中，可在变量浏览器中检查。默认使用合成 PSG；开启 `USE_REAL_DATA` 可读取真实 HSP，改 `DEVICE` 可切换 CPU/CUDA。此 notebook 验证主干数据流和梯度，不训练 SSL。

如果在 VS Code 中调试，选择 SleepWM Python kernel，可给单元格设置断点或使用 Debug Cell。更改配置后，从构建模型的单元格重新运行。

In [1]:
from pathlib import Path
import os
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import yaml
from dataclasses import replace
from backbone import BackboneConfig, MaskPlan, build_backbone
from dataloader import WindowDataset, collate_windows, as_signal_batch
from dataloader.synthetic import synthetic_windows

torch.manual_seed(7)
torch.set_num_threads(4)
print('Python:', sys.executable)
print('PyTorch:', torch.__version__, 'CUDA available:', torch.cuda.is_available())

Python: C:\Users\user\miniconda3\envs\SleepWM\python.exe
PyTorch: 2.12.1+cu132 CUDA available: True


## 1. 参数与数据源

首次调试保持小 batch。正式配置在 `configs/backbone/base.yaml`：patch 内 CNN + Transformer，patch 间 Transformer，cross-attention 融合。epoch 是读取单位，上下文块限制在一个 epoch 内。

In [2]:
USE_REAL_DATA = os.environ.get('PSG_NOTEBOOK_REAL', '0') == '1'
DATA_ROOT = Path(r'I:\HSP\I0002-preprocess\processed\v1.0.0')
DEVICE = os.environ.get('PSG_NOTEBOOK_DEVICE', 'cpu')
BATCH_SIZE = 2
EPOCHS = 2

if USE_REAL_DATA:
    with WindowDataset(DATA_ROOT, context_epochs=EPOCHS, tasks=()) as dataset:
        raw = collate_windows([dataset[i] for i in range(BATCH_SIZE)])
else:
    raw = synthetic_windows(batch_size=BATCH_SIZE, epochs=EPOCHS)

print('Reader signal shapes:', {k: tuple(v.shape) for k, v in raw['signals'].items()})
print('epoch_mask:', raw['epoch_mask'])
print('night_grade:', raw.get('night_grade'))

Reader signal shapes: {'eeg': (2, 2, 6, 6000), 'eog': (2, 2, 2, 6000), 'ecg': (2, 2, 1, 6000), 'emg': (2, 2, 3, 6000), 'respiratory': (2, 2, 3, 6000)}
epoch_mask: tensor([[ True,  True],
        [ True, False]])
night_grade: tensor([5, 5], dtype=torch.uint8)


## 2. Adapter → SignalBatch

将 SpO₂ 从 respiratory 拆出。QC、channel ids、单位与数值一起拆分；labels/subject 信息留在外层 `raw`。合成数据故意含缺失通道的 NaN 和尾部 padding。

`scales` 可传入固定的训练集尺度（例如每组除以一个已知常数），默认不改变物理数值。

In [3]:
batch = as_signal_batch(raw).to(DEVICE)
batch.validate()
for name, group in batch.groups.items():
    print(name, tuple(group.values.shape), group.channel_ids, group.units)
print('EEG valid [B,E,C]:', tuple(batch.groups['eeg'].valid.shape))
print('EEG channel_mask:', batch.groups['eeg'].channel_mask)

eeg (2, 2, 6, 6000) ('f3-m2', 'f4-m1', 'c3-m2', 'c4-m1', 'o1-m2', 'o2-m1') ('a.u.', 'a.u.', 'a.u.', 'a.u.', 'a.u.', 'a.u.')
eog (2, 2, 2, 6000) ('e1', 'e2') ('a.u.', 'a.u.')
ecg (2, 2, 1, 6000) ('ecg',) ('a.u.',)
emg (2, 2, 3, 6000) ('chin1-chin2', 'lat', 'rat') ('a.u.', 'a.u.', 'a.u.')
respiratory (2, 2, 2, 6000) ('airflow', 'snore') ('a.u.', 'a.u.')
spo2 (2, 2, 1, 6000) ('spo2',) ('a.u.',)
EEG valid [B,E,C]: (2, 2, 6)
EEG channel_mask: tensor([[ True,  True,  True,  True,  True,  True],
        [ True,  True,  True,  True,  True, False]])


## 3. 构建模型

`variant='direct_linear'` 可替换 patch 内网络；`blocks=()` 让序列编码成为 identity；`fusion='none'` 可关闭融合。禁用项必须同步调整后续请求的输出。

In [4]:
config = BackboneConfig.from_dict(yaml.safe_load((ROOT / 'configs/backbone/base.yaml').read_text()))
model = build_backbone(config, batch).to(DEVICE).eval()
signal = model.modality_encoders['eeg'].signal_encoder
print(model)
print('Parameters:', sum(p.numel() for p in model.parameters()))

PSGBackbone(
  (modality_encoders): ModuleDict(
    (eeg): ModalityEncoder(
      (signal_encoder): SignalEncoder(
        (patchifier): Patchifier()
        (masker): MaskApplier()
        (patch_encoder): PatchEncoder(
          (projection): Linear(in_features=20, out_features=64, bias=True)
          (blocks): ModuleList(
            (0-1): 2 x CNNSequenceBlock(
              (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
              (conv): Conv1d(64, 64, kernel_size=(3,), stride=(1,))
              (activation): GELU(approximate='none')
            )
            (2): TransformerSequenceBlock(
              (block): CrossAttentionBlock(
                (query_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
                (source_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
                (attention): Attention(
                  (query): Linear(in_features=64, out_features=64, bias=True)
                

## 4. Patchifier

`[B,E,C,S] → [B,C,N,L]`。逐 epoch 切片，保留 recording 时间坐标，不跨越缺口。

In [5]:
with torch.no_grad():
    patches = signal.patchifier(batch.groups['eeg'], batch)
print('patches:', tuple(patches.values.shape))
print('sample_visible:', tuple(patches.sample_visible.shape))
print('patch data_valid:', tuple(patches.data_valid.shape))
print('first patch intervals (seconds):', patches.layout.time_intervals_ns[0, :5].cpu() / 1e9)
assert torch.isfinite(patches.values).all()

patches: (2, 6, 60, 200)
sample_visible: (2, 6, 60, 200)
patch data_valid: (2, 6, 60)
first patch intervals (seconds): tensor([[0., 1.],
        [1., 2.],
        [2., 3.],
        [3., 4.],
        [4., 5.]])


## 5. PatchEncoder：查看每个网络块的输入

每个 patch 单独编码。CNN/Transformer 的输入轴为 `[B*C*N,P,H]`，同一个网络块从不读取另一个 patch。

In [ ]:
block_shapes = {}
handles = []
for i, block in enumerate(getattr(signal.patch_encoder, 'blocks', [])):
    def observe(module, inputs, key=f'patch_block_{i}'):
        block_shapes[key] = tuple(inputs[0].tokens.shape)
    handles.append(block.register_forward_pre_hook(observe))
try:
    with torch.no_grad():
        patch_grid = signal.patch_encoder(patches)
finally:
    for handle in handles:
        handle.remove()
print('architecture inputs:', block_shapes)
print('patch tokens:', tuple(patch_grid.tokens.shape))
assert patch_grid.tokens.shape[:3] == patches.data_valid.shape

## 6. Token 遮挡

示例隐藏每5个 token 中的第2个。`data_valid` 保留数据质量；`visible` 表示本次可见性；`target_mask` 属于外部任务。

In [ ]:
token_visible = torch.ones_like(patch_grid.visible)
token_visible[..., 1::5] = False
plan = MaskPlan('token', {'eeg': token_visible})
masked_grid = signal.masker(patch_grid, token_visible)
target_mask = ~token_visible
loss_mask = patch_grid.data_valid & target_mask
print('valid targets:', int(loss_mask.sum()))
print('context tokens:', int(masked_grid.active.sum()))
assert torch.equal(masked_grid.data_valid, patch_grid.data_valid)
assert torch.count_nonzero(masked_grid.tokens[~masked_grid.active]) == 0

## 7. PatchSequenceEncoder：30秒真实分块

网络输入 `[B*C*E*ceil(patches_per_epoch/Q),Q,D]`。不足一个块的尾部补齐并由 mask 排除；输出恢复为 `[B,C,N,D]`。

In [ ]:
sequence_shapes = {}
handles = []
for i, block in enumerate(signal.sequence_encoder.blocks):
    def observe(module, inputs, key=f'sequence_block_{i}'):
        sequence_shapes[key] = tuple(inputs[0].tokens.shape)
    handles.append(block.register_forward_pre_hook(observe))
try:
    with torch.no_grad():
        local = signal.sequence_encoder(masked_grid)
finally:
    for handle in handles:
        handle.remove()
print('architecture inputs:', sequence_shapes)
print('local:', tuple(local.tokens.shape))
print('position intervals (s):', local.time_intervals_ns[0, :4].cpu() / 1e9)
print('dependency intervals (s):', local.context_intervals_ns[0, :4].cpu() / 1e9)
print('available_at_ns:', local.available_at_ns[0, :4])  # -1: offline availability unknown

## 8. ChannelAggregator

通道身份编码之后沿 C 汇聚：`[B,C,N,D] → [B,N,D]`。全无效位置返回0；coverage 不因主动遮挡而改变。

In [ ]:
with torch.no_grad():
    eeg_features = model.modality_encoders['eeg'].channel_aggregator(local)
print('eeg features:', tuple(eeg_features.tokens.shape))
print('feature mask:', tuple(eeg_features.active.shape))
assert torch.isfinite(eeg_features.tokens).all()

## 9. 完整 backbone + Fusion

各模态都调用自己的 ModalityEncoder。第一版 Fusion 要求相同时间网格；cross-attention 在同一时间点的模态之间执行。

In [ ]:
with torch.no_grad():
    result = model(batch, plan, outputs=('patch_tokens', 'local', 'features', 'joint'))
    fused_manually = model.fusion(result.features)
print('features:', {k: tuple(v.tokens.shape) for k, v in result.features.items()})
print('joint:', tuple(result.joint.tokens.shape))
torch.testing.assert_close(result.local['eeg'].tokens, local.tokens)
torch.testing.assert_close(result.features['eeg'].tokens, eeg_features.tokens)
torch.testing.assert_close(result.joint.tokens, fused_manually.tokens)
assert torch.isfinite(result.joint.tokens).all()
print('manual steps == full forward: OK')

## 10. Waveform 遮挡

原始波形上的部分 patch 遮挡在投影之前执行，不改变 QC。改变这里的区域可逐点观察对 token 的影响。

In [ ]:
wave_visible = torch.ones_like(batch.groups['eeg'].values, dtype=torch.bool)
wave_visible[:, 0, :, :100] = False
wave_plan = MaskPlan('waveform', {'eeg': wave_visible})
with torch.no_grad():
    wave_result = model(batch, wave_plan, outputs=('patch_tokens', 'local'))
print('waveform visibility:', tuple(wave_visible.shape))
print('waveform-masked tokens:', tuple(wave_result.patch_tokens['eeg'].tokens.shape))
assert torch.isfinite(wave_result.local['eeg'].tokens).all()

## 11. 防泄漏检查

将 token 遮挡位置的原始值改成 NaN，整个融合输出应保持一致。检查不会修改原始 batch。

In [ ]:
group = batch.groups['eeg']
b, e, c, s = group.values.shape
p = patches.layout.patches_per_epoch
hidden_samples = (~token_visible).reshape(b,c,e,p).permute(0,2,1,3).repeat_interleave(patches.layout.patch_samples, -1)
changed_values = group.values.clone().masked_fill(hidden_samples, float('nan'))
changed_batch = replace(batch, groups={**batch.groups, 'eeg': replace(group, values=changed_values)})
with torch.no_grad():
    changed_result = model(changed_batch, plan, outputs=('joint',))
torch.testing.assert_close(changed_result.joint.tokens, result.joint.tokens)
print('hidden raw values cannot change visible representation: OK')

## 12. 单步梯度与优化器

这里只用一个小随机目标检查反向传播与参数更新。它不是 JEPA、对比学习或正式训练目标。

In [ ]:
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
optimizer.zero_grad(set_to_none=True)
train_output = model(batch, outputs=('features',)).features['eeg']
assert train_output.active.any(), 'This window has no valid EEG; choose another window.'
selected = train_output.tokens[train_output.active]
toy_target = torch.randn_like(selected)
loss = (selected - toy_target).square().mean()
loss.backward()
grads = [p.grad for p in model.parameters() if p.grad is not None]
assert grads and all(torch.isfinite(g).all() for g in grads)
assert any(g.abs().sum() > 0 for g in grads)
optimizer.step()
model.eval()
print('toy loss:', float(loss.detach()), 'finite gradients:', len(grads))

## 13. 三类预训练的 shape 接入检查

只检查外部 head 能接通主干：JEPA predictor 使用可见上下文产生目标位置表示；对比学习对窗口表示投影；masked-code 对每个目标位置输出 K 类 logits。这些小随机 head 不属于 backbone，也不是已训练的预训练方法。

In [ ]:
with torch.no_grad():
    features = model(batch, outputs=('features', 'local'))
    sequence = features.features['eeg']
    weight = sequence.active.to(sequence.tokens.dtype)
    pooled = (sequence.tokens * weight[...,None]).sum(1) / weight.sum(1,keepdim=True).clamp_min(1)
    target_count, codebook_size, projection_dim = 4, 32, 16
    jepa_predictor = torch.nn.Linear(config.feature_dim, target_count * config.feature_dim).to(DEVICE)
    projector = torch.nn.Linear(config.feature_dim, projection_dim).to(DEVICE)
    code_head = torch.nn.Linear(config.feature_dim, codebook_size).to(DEVICE)
    predicted_targets = jepa_predictor(pooled).reshape(BATCH_SIZE,target_count,config.feature_dim)
    contrastive_z = projector(pooled)
    code_logits = code_head(features.local['eeg'].tokens)
print('JEPA target predictions [B,N_target,D]:', tuple(predicted_targets.shape))
print('contrastive projections [B,Z]:', tuple(contrastive_z.shape))
print('code logits [B,C,N,K]:', tuple(code_logits.shape))

## 14. 保存与重新加载

只保存本次 notebook 的主干配置、输入规格和 state_dict；不保存原始数据。配置保留通道顺序、采样率、单位/尺度和网络选择。

In [ ]:
CHECKPOINT = ROOT / 'artifacts' / 'backbone' / ('stepwise_real.pt' if USE_REAL_DATA else 'stepwise_synthetic.pt')
CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
torch.save({'config': config.to_dict(), 'input_spec': model.input_spec, 'state_dict': model.state_dict()}, CHECKPOINT)
saved = torch.load(CHECKPOINT, map_location=DEVICE, weights_only=True)
restored = build_backbone(BackboneConfig.from_dict(saved['config']), saved['input_spec']).to(DEVICE).eval()
restored.load_state_dict(saved['state_dict'])
with torch.no_grad():
    expected = model(batch, outputs=('joint',)).joint.tokens
    actual = restored(batch, outputs=('joint',)).joint.tokens
torch.testing.assert_close(actual, expected)
print('reload: OK', CHECKPOINT)
print('All stepwise checks passed.')